# Calibración Automatizada de Sensores de Fuerza (RH56DFTP-2R)
Este script realiza una calibración automatizada y segura de los sensores de fuerza de la mano robótica Inspire RH56DFTP-2R.
Para evitar que los dedos colisionen fuertemente o queden con tensión residual en sus resortes, el código orquesta un posicionamiento preventivo antes de inyectar el comando de calibración en el registro Modbus `1009`.
## Procedimiento de la Rutina
1. **Apertura total**: Extiende todos los dedos al máximo (`1000`) para liberar tensiones.
2. **Posición de cero estrés**: Cierra los dedos principales (`0`), flexiona casi por completo el pulgar (`100`) y deja la rotación en un punto de descanso seguro (`990`).
3. **Disparo de calibración**: Envía el comando `[1]` al registro `1009`.
4. **Tiempo de espera**: Otorga 7 segundos para que el hardware procese la lectura y encere los sensores físicos.
5. **Reestablecimiento**: Abre la mano por seguridad antes de cerrar el socket TCP.

## Código Fuente

In [1]:
import time
from inspire_rh56dftp import InspireHand

IP_MANO = "192.168.123.211"
REG_CALIBRAR_FUERZA = 1009
REG_FORCE_SET = 1498
REG_ANGLE_SET = 1486

def main():
    print(f"Conectando a la mano en {IP_MANO} ...")
    mano = InspireHand(ip=IP_MANO)

    try:
        print(f"Enviando comando de calibración de fuerza al registro {REG_CALIBRAR_FUERZA}...")
        mano._write_and_ack(REG_FORCE_SET, [200,200,200,200,200,200])
        mano._write_and_ack(REG_ANGLE_SET, [1000,1000,1000,1000,1000,1000])
        time.sleep(1.5)
        mano._write_and_ack(REG_ANGLE_SET, [0,0,0,0,0,1000])
        time.sleep(1.5)
        mano._write_and_ack(REG_CALIBRAR_FUERZA, [1])
        
        print("Esperando a que la mano procese la calibración (asegúrate de que los dedos estén libres)...")
        time.sleep(6.0)
        
        print("✓ Calibración de fuerza finalizada.")

    except Exception as e:
        print(f"⚠ Error durante la calibración: {e}")
    finally:
        mano._write_and_ack(REG_ANGLE_SET, [1000,1000,1000,1000,1000,1000])
        time.sleep(1.0)
        mano.close()
        print("Conexión TCP cerrada.")

if __name__ == "__main__":
    main()

Conectando a la mano en 192.168.123.211 ...
Enviando comando de calibración de fuerza al registro 1009...
Esperando a que la mano procese la calibración (asegúrate de que los dedos estén libres)...
✓ Calibración de fuerza finalizada.
Conexión TCP cerrada.
